# Rendimiento: Python puro frente a Numba y algoritmos

1. Tiempo por barrido de Metrópolis en Python puro y compilado con `@njit`.
2. Costo por **muestra independiente** (tiempo por paso × 2 τ_int) de
   Metrópolis, Wolff e híbrido cerca de T_c: es la cifra que importa.

In [ ]:
import os, sys
RAIZ = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, RAIZ)
os.chdir(RAIZ)
import time
import numpy as np
import matplotlib.pyplot as plt
import parametros
from isingmc.flujo import config_desde_modulo, preparar
from isingmc.red import construir_libreta
from isingmc.montecarlo import barrido_metropolis, barrido_temperaturas
from isingmc.analisis import tau_int

cfg = config_desde_modulo(parametros)
prep = preparar(cfg, verbose=False)
enl, nat = prep["enl"], prep["nat"]

## 1. Python puro frente a Numba (mismo algoritmo)

In [ ]:
def barrido_metropolis_py(spins, vec, Jv, nvec, beta):
    N = spins.size
    for _ in range(N):
        i = np.random.randint(N)
        h = 0.0
        for c in range(nvec[i]):
            h += Jv[i, c] * spins[vec[i, c]]
        dE = 2.0 * spins[i] * h
        if dE <= 0.0 or np.random.random() < np.exp(-beta * dE):
            spins[i] = -spins[i]

beta = 1.0 / (cfg["k_B"] * 600.0)
filas = []
for L in (3, 6, 9, 12):
    vec, Jv, nvec = construir_libreta(L, cfg["aspecto"] * L, nat, enl)
    N = len(nvec)
    spins = np.random.choice([-1, 1], size=N).astype(np.int64)
    barrido_metropolis(spins, vec, Jv, nvec, beta)          # compilación
    t0 = time.perf_counter(); [barrido_metropolis_py(spins, vec, Jv, nvec, beta) for _ in range(3)]
    t_py = (time.perf_counter() - t0) / 3
    t0 = time.perf_counter(); [barrido_metropolis(spins, vec, Jv, nvec, beta) for _ in range(300)]
    t_nb = (time.perf_counter() - t0) / 300
    filas.append((N, t_py, t_nb))
    print(f"N={N:6d}  Python={t_py*1e3:9.2f} ms  Numba={t_nb*1e3:7.3f} ms  x{t_py/t_nb:5.0f}")

## 2. Costo por muestra independiente cerca de T_c

In [ ]:
T = 590.0      # cerca de T_c de la desnuda (Ising); cambie según el material
L = 9
vec, Jv, nvec = construir_libreta(L, cfg["aspecto"] * L, nat, enl)
eps = np.ones(len(nvec), dtype=np.int64)
for alg in ("metropolis", "wolff", "hibrido"):
    t0 = time.perf_counter()
    E, M, _ = barrido_temperaturas(vec, Jv, nvec, eps, [T], cfg["k_B"], 1000, 1, 4000, alg, seed=1)
    dt = (time.perf_counter() - t0) / 5000
    tau = max(tau_int(E[0]), tau_int(M[0] ** 2))
    print(f"{alg:10s} {dt*1e3:6.2f} ms/paso  tau_int = {tau:6.1f}  -> {2*tau*dt*1e3:7.1f} ms por muestra independiente")